# Notebook 2: Expanded Data Validation, Model Selection, and Error Analysis

This notebook is the next-stage modeling notebook after Notebook 1.5.

Notebook 1.5 focused on:
- getting the cleaned CFD dataset into the pipeline
- establishing a stronger baseline model
- tuning the best tree-based model
- exporting a deployable `.joblib` artifact

Notebook 2 is designed for the next phase of the project, once more CFD data has been added.

## Main goals of Notebook 2
1. load an expanded CFD dataset
2. validate the dataset more strictly before modeling
3. use more robust validation logic, including group-aware evaluation when possible
4. compare multiple model families and choose the best overall model family
5. tune the winning family
6. inspect learning behavior and error patterns
7. export a new model artifact plus metadata for app integration

## Important note
This notebook assumes the dataset is already cleaned.  
It should validate the data, not silently rewrite or relabel it.

## 1. Imports and project paths

This cell imports the libraries used for:
- data loading and validation
- preprocessing
- model comparison
- model tuning
- group-aware evaluation
- learning curves
- error analysis
- artifact export

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    GroupShuffleSplit,
    KFold,
    RepeatedKFold,
    cross_validate,
    learning_curve,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 5)

PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]

CSV_CANDIDATES = []
for root in PROJECT_ROOT_CANDIDATES:
    CSV_CANDIDATES.extend([
        root / "data" / "processed" / "openfoam_phase2_expanded.csv",
        root / "data" / "processed" / "openfoam_master_expanded.csv",
        root / "data" / "processed" / "openfoam_phase1_cleaned.csv",
        root / "data" / "processed" / "openFoamSimulationData4 - Sheet1.csv",
    ])

MODEL_DIR = None
for root in PROJECT_ROOT_CANDIDATES:
    candidate = root / "models"
    if candidate.parent.exists():
        MODEL_DIR = candidate
        break

if MODEL_DIR is None:
    MODEL_DIR = Path.cwd() / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

CSV_CANDIDATES

## 2. Load the expanded dataset

This notebook looks first for an expanded phase 2 style CFD dataset.  
If that does not exist yet, it will fall back to the current cleaned OpenFOAM dataset so the notebook can still be tested.

In [ ]:
CSV_PATH = None
for candidate in CSV_CANDIDATES:
    if candidate.exists():
        CSV_PATH = candidate
        break

if CSV_PATH is None:
    raise FileNotFoundError(
        "Could not find the expanded CFD dataset. "
        "Place it in data/processed/ or update CSV_PATH manually."
    )

df = pd.read_csv(CSV_PATH)
print(f"Loaded dataset from: {CSV_PATH}")
print(f"Shape: {df.shape}")
df.head()

## 3. Strict dataset validation

This section checks:
- required target column
- data types
- missing values
- duplicate rows
- target scale
- category consistency

Notebook 2 should assume the dataset is already cleaned and should validate it rather than rewrite it.

In [ ]:
TARGET_COLUMN = "separation_x_over_c"

print("Columns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", int(df.duplicated().sum()))

if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Target column '{TARGET_COLUMN}' not found.")

target_min = df[TARGET_COLUMN].min()
target_max = df[TARGET_COLUMN].max()
print(f"\nTarget range before any correction check: min={target_min}, max={target_max}")

if target_max > 1.0 and target_max <= 100.0:
    print("Target appears to be in percent form. Converting to 0 to 1 x/c scale.")
    df[TARGET_COLUMN] = df[TARGET_COLUMN] / 100.0

print(
    f"Target range after correction check: "
    f"min={df[TARGET_COLUMN].min()}, max={df[TARGET_COLUMN].max()}"
)

assert df[TARGET_COLUMN].between(0, 1).all(), "Target contains values outside [0, 1]."

## 4. Lock the schema and validate categories

This notebook keeps the current V1/V1.5 feature schema for app compatibility.

Optional metadata columns such as:
- `data_source`
- `configuration_id`
- `geometry_id`
- `source_case`

may exist in the dataset, but they are not included as model inputs unless intentionally added later.

In [ ]:
CANDIDATE_FEATURE_COLUMNS = [
    "airfoil_family",
    "tubercle_amplitude",
    "tubercle_wavelength",
    "tubercle_shape",
    "root_chord",
    "tip_chord",
    "sweep_angle",
    "angle_of_attack",
    "airspeed",
]

missing_candidates = [col for col in CANDIDATE_FEATURE_COLUMNS if col not in df.columns]
if missing_candidates:
    print("Missing candidate feature columns:", missing_candidates)

present_feature_columns = [col for col in CANDIDATE_FEATURE_COLUMNS if col in df.columns]

constant_columns = [
    col for col in present_feature_columns
    if df[col].nunique(dropna=False) <= 1
]

FEATURE_COLUMNS = [col for col in present_feature_columns if col not in constant_columns]

print("Present feature columns:", present_feature_columns)
print("Dropped constant columns:", constant_columns)
print("Final feature columns used for modeling:", FEATURE_COLUMNS)

VALID_AIRFOIL_FAMILIES = {"symmetric", "cambered", "biomimetic"}
VALID_TUBERCLE_SHAPES = {"none", "whale", "biomimetic_v1"}

if "airfoil_family" in df.columns:
    observed_airfoil_families = set(df["airfoil_family"].dropna().astype(str).str.lower().unique())
    print("Observed airfoil_family values:", observed_airfoil_families)
    print("Unexpected airfoil_family values:", observed_airfoil_families - VALID_AIRFOIL_FAMILIES)

if "tubercle_shape" in df.columns:
    observed_tubercle_shapes = set(df["tubercle_shape"].dropna().astype(str).str.lower().unique())
    print("Observed tubercle_shape values:", observed_tubercle_shapes)
    print("Unexpected tubercle_shape values:", observed_tubercle_shapes - VALID_TUBERCLE_SHAPES)

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

## 5. Build an evaluation group column

As the dataset grows, multiple rows may come from closely related simulation configurations.
Notebook 2 tries to avoid overly optimistic evaluation by using grouped validation when possible.

Priority for grouping:
1. use an explicit existing ID column if available
2. otherwise build a synthetic grouping key from stable geometry fields

In [ ]:
GROUP_CANDIDATE_COLUMNS = [
    "configuration_id",
    "geometry_id",
    "source_case",
    "profile_id",
]

group_column_used = None
groups = None

for candidate in GROUP_CANDIDATE_COLUMNS:
    if candidate in df.columns:
        group_column_used = candidate
        groups = df[candidate].astype(str)
        break

if groups is None:
    synthetic_group_cols = [
        col for col in [
            "airfoil_family",
            "tubercle_amplitude",
            "tubercle_wavelength",
            "tubercle_shape",
            "root_chord",
            "tip_chord",
            "sweep_angle",
        ] if col in df.columns
    ]

    if synthetic_group_cols:
        temp = df[synthetic_group_cols].copy()
        for col in temp.columns:
            if pd.api.types.is_numeric_dtype(temp[col]):
                temp[col] = temp[col].round(4).astype(str)
            else:
                temp[col] = temp[col].astype(str).str.lower()
        groups = temp.apply(lambda row: "|".join(row.values.astype(str)), axis=1)
        group_column_used = "synthetic_group_key"

print("Grouping strategy:", group_column_used)
if groups is not None:
    print("Number of unique groups:", groups.nunique())

## 6. Separate numeric and categorical features

In [ ]:
POTENTIAL_CATEGORICAL = ["airfoil_family", "tubercle_shape"]
categorical_features = [col for col in FEATURE_COLUMNS if col in POTENTIAL_CATEGORICAL]
numeric_features = [col for col in FEATURE_COLUMNS if col not in categorical_features]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 7. Quick dataset inspection plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(y, bins=15)
axes[0].set_xlabel("separation_x_over_c")
axes[0].set_ylabel("Count")
axes[0].set_title("Target Distribution")

if "airfoil_family" in df.columns:
    family_counts = df["airfoil_family"].astype(str).value_counts()
    axes[1].bar(family_counts.index, family_counts.values)
    axes[1].set_title("Rows by Airfoil Family")
    axes[1].set_ylabel("Count")
else:
    axes[1].text(0.5, 0.5, "No airfoil_family column", ha="center", va="center")
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()

## 8. Create a held-out test split

If grouping information is available and there are enough unique groups, use `GroupShuffleSplit`
so the test rows come from unseen groups. Otherwise, fall back to a standard random split.

This is one of the major Notebook 2 improvements over Notebook 1.5.

In [ ]:
use_grouped_holdout = groups is not None and groups.nunique() >= 5

if use_grouped_holdout:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))
    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()
    y_train = y.iloc[train_idx].copy()
    y_test = y.iloc[test_idx].copy()
    groups_train = groups.iloc[train_idx].copy()
    print("Using grouped holdout split.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
    )
    groups_train = None
    print("Using standard random holdout split.")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## 9. Build the preprocessing pipeline

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

preprocessor

## 10. Compare candidate models using the best available validation strategy

Notebook 2 expands the model comparison stage.

Candidate models:
- Linear Regression
- Ridge Regression
- Elastic Net
- Random Forest
- Extra Trees
- Gradient Boosting

If grouped training data is available, use `GroupKFold`.  
Otherwise, use repeated K-fold cross-validation.

In [ ]:
candidate_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Elastic Net": ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42),
    "Random Forest": RandomForestRegressor(
        random_state=42,
        n_estimators=300,
        n_jobs=-1,
    ),
    "Extra Trees": ExtraTreesRegressor(
        random_state=42,
        n_estimators=300,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42,
    ),
}

if groups_train is not None and groups_train.nunique() >= 5:
    cv = GroupKFold(n_splits=min(5, groups_train.nunique()))
    grouped_cv = True
    print("Using GroupKFold for model comparison.")
else:
    cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)
    grouped_cv = False
    print("Using RepeatedKFold for model comparison.")

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

cv_rows = []

for model_name, estimator in candidate_models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )

    if grouped_cv:
        scores = cross_validate(
            pipeline,
            X_train,
            y_train,
            cv=cv,
            groups=groups_train,
            scoring=scoring,
            n_jobs=-1,
            return_train_score=False,
        )
    else:
        scores = cross_validate(
            pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
            return_train_score=False,
        )

    cv_rows.append({
        "model": model_name,
        "cv_mae_mean": -scores["test_mae"].mean(),
        "cv_mae_std": scores["test_mae"].std(),
        "cv_rmse_mean": -scores["test_rmse"].mean(),
        "cv_rmse_std": scores["test_rmse"].std(),
        "cv_r2_mean": scores["test_r2"].mean(),
        "cv_r2_std": scores["test_r2"].std(),
    })

cv_results_df = pd.DataFrame(cv_rows).sort_values(by="cv_rmse_mean")
cv_results_df

## 11. Choose the best overall model family to tune

Notebook 1.5 intentionally focused on tuning the best tree-based family.
Notebook 2 now selects the best overall family from the comparison table, then tunes that family.

RMSE remains the primary selection metric.

In [ ]:
best_model_name_pre_tuning = cv_results_df.iloc[0]["model"]
print("Best overall model before tuning:", best_model_name_pre_tuning)

## 12. Tune the best overall model family

This section tunes the winning model family with a parameter grid appropriate for that family.
Not every model needs the same tuning strategy.

If Linear Regression wins, the notebook keeps the default pipeline because there is little to tune.

In [ ]:
best_params = {}

if best_model_name_pre_tuning == "Linear Regression":
    best_model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LinearRegression()),
        ]
    )
    best_model.fit(X_train, y_train)

elif best_model_name_pre_tuning == "Ridge Regression":
    tuning_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", Ridge()),
        ]
    )

    param_grid = {
        "model__alpha": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    }

    grid_search = GridSearchCV(
        estimator=tuning_pipeline,
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1,
        verbose=0,
    )

    if grouped_cv:
        grid_search.fit(X_train, y_train, groups=groups_train)
    else:
        grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

elif best_model_name_pre_tuning == "Elastic Net":
    tuning_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", ElasticNet(random_state=42, max_iter=10000)),
        ]
    )

    param_grid = {
        "model__alpha": [0.0001, 0.001, 0.01, 0.1, 1.0],
        "model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
    }

    grid_search = GridSearchCV(
        estimator=tuning_pipeline,
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1,
        verbose=0,
    )

    if grouped_cv:
        grid_search.fit(X_train, y_train, groups=groups_train)
    else:
        grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

elif best_model_name_pre_tuning == "Random Forest":
    tuning_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),
        ]
    )

    param_grid = {
        "model__n_estimators": [200, 400, 600],
        "model__max_depth": [None, 3, 5, 8],
        "model__min_samples_split": [2, 4, 6],
        "model__min_samples_leaf": [1, 2, 3],
        "model__max_features": [1.0, "sqrt"],
    }

    grid_search = GridSearchCV(
        estimator=tuning_pipeline,
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1,
        verbose=0,
    )

    if grouped_cv:
        grid_search.fit(X_train, y_train, groups=groups_train)
    else:
        grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

elif best_model_name_pre_tuning == "Extra Trees":
    tuning_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", ExtraTreesRegressor(random_state=42, n_jobs=-1)),
        ]
    )

    param_grid = {
        "model__n_estimators": [200, 400, 600],
        "model__max_depth": [None, 3, 5, 8],
        "model__min_samples_split": [2, 4, 6],
        "model__min_samples_leaf": [1, 2, 3],
        "model__max_features": [1.0, "sqrt"],
    }

    grid_search = GridSearchCV(
        estimator=tuning_pipeline,
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1,
        verbose=0,
    )

    if grouped_cv:
        grid_search.fit(X_train, y_train, groups=groups_train)
    else:
        grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

elif best_model_name_pre_tuning == "Gradient Boosting":
    tuning_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", GradientBoostingRegressor(random_state=42)),
        ]
    )

    param_grid = {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__max_depth": [2, 3, 4],
        "model__min_samples_split": [2, 4, 6],
        "model__min_samples_leaf": [1, 2, 3],
        "model__subsample": [0.8, 1.0],
    }

    grid_search = GridSearchCV(
        estimator=tuning_pipeline,
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1,
        verbose=0,
    )

    if grouped_cv:
        grid_search.fit(X_train, y_train, groups=groups_train)
    else:
        grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

else:
    raise ValueError(f"Unexpected model name: {best_model_name_pre_tuning}")

print("Chosen model family:", best_model_name_pre_tuning)
print("Best parameters:", best_params)

## 13. Evaluate the tuned model on the held-out test set

In [ ]:
test_preds = best_model.predict(X_test)

test_mae = mean_absolute_error(y_test, test_preds)
test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
test_r2 = r2_score(y_test, test_preds)

results_df = pd.DataFrame(
    [
        {
            "model": best_model_name_pre_tuning,
            "MAE": test_mae,
            "RMSE": test_rmse,
            "R2": test_r2,
        }
    ]
).sort_values(by="RMSE")

results_df

## 14. Interpret the hold-out metrics

Use RMSE as the primary deciding metric, with MAE and R² as supporting context.
The hold-out metrics matter because they show how the tuned model performs on unseen rows or groups.

In [ ]:
best_model_name = results_df.iloc[0]["model"]
safe_model_name = best_model_name.lower().replace(" ", "_")
best_model_filename = f"notebook2_{safe_model_name}.joblib"

print("Best model selected:", best_model_name)
print("Filename:", best_model_filename)
print("Hold-out RMSE:", float(results_df.iloc[0]["RMSE"]))

## 15. Plot actual vs predicted values

In [ ]:
plt.scatter(y_test, test_preds)
min_val = min(y_test.min(), test_preds.min())
max_val = max(y_test.max(), test_preds.max())
plt.plot([min_val, max_val], [min_val, max_val])
plt.xlabel("Actual separation_x_over_c")
plt.ylabel("Predicted separation_x_over_c")
plt.title(f"Tuned {best_model_name}: Actual vs Predicted")
plt.show()

## 16. Residual analysis

Notebook 2 adds a more explicit residual check so it is easier to see whether
the model is systematically overpredicting or underpredicting.

In [ ]:
residuals = y_test - test_preds

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(test_preds, residuals)
axes[0].axhline(0, linestyle="--")
axes[0].set_xlabel("Predicted separation_x_over_c")
axes[0].set_ylabel("Residual (actual - predicted)")
axes[0].set_title("Residuals vs Predicted")

axes[1].hist(residuals, bins=12)
axes[1].set_title("Residual Distribution")
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## 17. Error analysis by airfoil family

If `airfoil_family` exists in the test set, summarize average errors by family.
This helps identify whether one family is currently harder for the model.

In [ ]:
error_analysis_df = X_test.copy()
error_analysis_df["actual"] = y_test.values
error_analysis_df["predicted"] = test_preds
error_analysis_df["absolute_error"] = np.abs(error_analysis_df["actual"] - error_analysis_df["predicted"])
error_analysis_df["residual"] = error_analysis_df["actual"] - error_analysis_df["predicted"]

if "airfoil_family" in error_analysis_df.columns:
    family_error_summary = (
        error_analysis_df.groupby("airfoil_family")[["absolute_error", "residual"]]
        .agg(["mean", "median", "count"])
    )
    family_error_summary
else:
    print("No airfoil_family column available for grouped error analysis.")

## 18. Learning curve

This helps answer a key Notebook 2 question:
Would more data likely improve the model?

If validation scores are still improving as training size grows, that suggests more data is likely valuable.

In [ ]:
train_sizes, train_scores, valid_scores = learning_curve(
    estimator=best_model,
    X=X_train,
    y=y_train,
    cv=cv,
    groups=groups_train if grouped_cv else None,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    train_sizes=np.linspace(0.3, 1.0, 5),
)

train_rmse = -train_scores.mean(axis=1)
valid_rmse = -valid_scores.mean(axis=1)

plt.plot(train_sizes, train_rmse, marker="o", label="Train RMSE")
plt.plot(train_sizes, valid_rmse, marker="o", label="Validation RMSE")
plt.xlabel("Training set size")
plt.ylabel("RMSE")
plt.title("Learning Curve")
plt.legend()
plt.show()

## 19. Feature importance analysis

For the tuned model, compute permutation importance on the hold-out set.

In [ ]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=20,
    random_state=42,
    n_jobs=-1,
)

feature_importance_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values(by="importance_mean", ascending=False)

feature_importance_df

## 20. Plot feature importances

In [ ]:
plt.bar(feature_importance_df["feature"], feature_importance_df["importance_mean"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Permutation Importance")
plt.title("Feature Importance for Notebook 2 Model")
plt.tight_layout()
plt.show()

## 21. Save the tuned model artifact

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
best_model_path = MODEL_DIR / best_model_filename

joblib.dump(best_model, best_model_path)

print("Saved model to:", best_model_path)
print("Exists:", best_model_path.exists())

## 22. Save predictions for review

This export is useful for debugging and for manual review of which rows are hardest for the model.

In [ ]:
predictions_export = error_analysis_df.copy()
predictions_export_path = MODEL_DIR / f"notebook2_{safe_model_name}_holdout_predictions.csv"
predictions_export.to_csv(predictions_export_path, index=False)

print("Saved holdout predictions to:", predictions_export_path)

## 23. Save a metadata summary

In [ ]:
metadata = {
    "dataset_path": str(CSV_PATH),
    "target_column": TARGET_COLUMN,
    "feature_columns_used": FEATURE_COLUMNS,
    "dropped_constant_columns": constant_columns,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "group_column_used": group_column_used,
    "grouped_cv_used": grouped_cv,
    "best_model_name_pre_tuning": best_model_name_pre_tuning,
    "best_model_filename": best_model_filename,
    "best_params": best_params,
    "holdout_metrics": {
        "mae": float(test_mae),
        "rmse": float(test_rmse),
        "r2": float(test_r2),
    },
}

metadata_path = MODEL_DIR / f"notebook2_{safe_model_name}_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved metadata to:", metadata_path)

## 24. Quick inference schema check

This cell verifies that a one-row example in the app schema can be passed into the saved pipeline.

In [ ]:
example_input = pd.DataFrame(
    [
        {
            feature: value for feature, value in {
                "airfoil_family": "biomimetic",
                "tubercle_amplitude": 26.247,
                "tubercle_wavelength": 49.607,
                "tubercle_shape": "whale",
                "root_chord": 1.0,
                "tip_chord": 1.0,
                "sweep_angle": 0.0,
                "angle_of_attack": 10.0,
                "airspeed": 30.0,
            }.items()
            if feature in FEATURE_COLUMNS
        }
    ]
)

example_prediction = best_model.predict(example_input)[0]
print(f"Example predicted separation_x_over_c: {example_prediction:.4f}")